## 0) Import Libraries

In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap
import warnings
warnings.filterwarnings('ignore')

## 1) Initial Exploration

In [ ]:
florence = pd.read_csv('Data/listings_florence.csv')
milan    = pd.read_csv('Data/listings_milan.csv')
rome     = pd.read_csv('Data/listings_rome.csv')

for name, df in [('Florence', florence), ('Milan', milan), ('Rome', rome)]:
    columns = list(df.columns)
    print(f'--- {name} ---')
    print(f'Listings: {len(df)}')
    print(f'Room types:\n{df["room_type"].value_counts()}')
    print()

columns_f, columns_m, columns_r = set(florence.columns), set(milan.columns), set(rome.columns)
if columns_f == columns_m == columns_r:
    print("All datasets have the same columns.\n")
else:
    print("Datasets have different columns.")

## 2) Visualization of Listings on the Map

In [ ]:
# Combine all cities with a label
florence['city'] = 'Florence'
milan['city']    = 'Milan'
rome['city']     = 'Rome'
all_listings = pd.concat([florence, milan, rome], ignore_index=True)

# Drop rows with missing coordinates
all_listings = all_listings.dropna(subset=['latitude', 'longitude'])

# Build map centered on Italy
m = folium.Map(location=[42.5, 12.5], zoom_start=6, tiles='CartoDB positron')

city_colors = {'Florence': 'red', 'Milan': 'blue', 'Rome': 'green'}
cluster = MarkerCluster().add_to(m)

for _, row in all_listings.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2,
        color=city_colors[row['city']],
        fill=True,
        fill_opacity=0.6,
        popup=f"{row['city']} | {row['room_type']} | €{row['price']}"
    ).add_to(cluster)

m